# 🎬 Colab-Comerza  📦 **v1.0.3**

Convierte un diseño estático (JSON) en un video animado 9:16 con locución y música.

---
### ⚡ Antes de empezar
`Runtime → Change runtime type → T4 GPU`

Luego ejecuta todas las celdas con `Runtime → Run all`

In [ ]:
# ============================================================
# 1. CLONAR REPO E INSTALAR DEPENDENCIAS
# ============================================================
import os, sys, json, re, io, base64

# Clonar repo
REPO = "https://github.com/alexdechile/colab-comerza"
if not os.path.exists("colab-comerza"):
    !git clone {REPO}
%cd colab-comerza

# Instalar dependencias (con salida visible para debug)
%pip install gtts moviepy pillow

# Descargar fuentes Montserrat
!wget -q "https://github.com/google/fonts/raw/main/ofl/montserrat/Montserrat-Regular.ttf" -O Montserrat-Regular.ttf
!wget -q "https://github.com/google/fonts/raw/main/ofl/montserrat/Montserrat-Bold.ttf" -O Montserrat-Bold.ttf
!wget -q "https://github.com/google/fonts/raw/main/ofl/montserrat/Montserrat-ExtraBold.ttf" -O Montserrat-ExtraBold.ttf
print("✅ Setup completo")


In [ ]:
# ============================================================
# 2. IMPORTAR LIBRERÍAS
# ============================================================
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from gtts import gTTS
from moviepy.editor import AudioFileClip, ImageSequenceClip, CompositeAudioClip
from moviepy.audio.AudioClip import AudioArrayClip

import torch
print("📦 Colab-Comerza v1.0.3")
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("✅ Librerías importadas")


In [ ]:
# ============================================================
# 3. CARGAR DISEÑO
# ============================================================
with open("mi-diseno.json") as f:
    design = json.load(f)

CANVAS_W = design["canvasSize"]["width"]   # 595
CANVAS_H = design["canvasSize"]["height"]  # 842
OUT_W, OUT_H = 1080, 1920
SCALE = OUT_W / CANVAS_W

def strip_html(text):
    return re.sub(r'<[^>]+>', '', text).strip()

def parse_color(c):
    if c.startswith("#"):
        c = c.lstrip("#")
        if len(c) == 6:
            return tuple(int(c[i:i+2], 16) for i in (0, 2, 4)) + (255,)
    m = re.findall(r'[\d.]+', c)
    if m:
        return tuple(int(float(x)) for x in m[:3]) + (255,)
    return (0, 0, 0, 255)

text_elements = []
img_info = None

for el in design["elements"]:
    if el["type"] == "image":
        img_info = {
            "x": el["x"] * SCALE,
            "y": el["y"] * SCALE,
            "w": el["width"] * SCALE,
            "h": el["height"] * SCALE,
        }
    elif el["type"] == "text":
        content = strip_html(el["content"])
        if not content:
            continue
        scale_y = el.get("scaleY", 1.0) or 1.0
        text_elements.append({
            "content": content,
            "x": el["x"] * SCALE,
            "y": el["y"] * SCALE,
            "font_size": int(el["fontSize"] * SCALE * scale_y),
            "color": el["color"],
            "has_shadow": el.get("hasShadow", False),
            "shadow_ox": el.get("shadowOffsetX", 0) * SCALE,
            "shadow_oy": el.get("shadowOffsetY", 0) * SCALE,
            "shadow_color": el.get("shadowColor", "rgba(0,0,0,0.5)"),
            "align": el.get("alignment", "left"),
            "width": el.get("width", CANVAS_W) * SCALE,
            "weight": el.get("fontWeight", 400),
        })
        print(f"  Texto: '{content}' | tamaño={text_elements[-1]['font_size']}px | pos=({int(text_elements[-1]['x'])}, {int(text_elements[-1]['y'])})")

print(f"✅ Diseño cargado: {len(text_elements)} textos, escala={SCALE:.2f}")


In [ ]:
# ============================================================
# 4. PREPARAR FONDO (imagen)
# ============================================================
bg = Image.open("imagen.png").convert("RGBA")
img_target_w = int(img_info["w"]) if img_info else OUT_W
img_target_h = int(img_info["h"]) if img_info else OUT_H
bg_resized = bg.resize((img_target_w, img_target_h), Image.LANCZOS)

base_frame = Image.new("RGBA", (OUT_W, OUT_H), (255, 255, 255, 255))
if img_info:
    base_frame.paste(bg_resized, (int(img_info["x"]), int(img_info["y"])), bg_resized)
print(f"✅ Fondo listo: {img_target_w}x{img_target_h}")


In [ ]:
# ============================================================
# 5. CARGAR FUENTES
# ============================================================
font_cache = {}

def get_font(weight, size):
    if weight >= 800:
        path = "Montserrat-ExtraBold.ttf"
    elif weight >= 700:
        path = "Montserrat-Bold.ttf"
    else:
        path = "Montserrat-Regular.ttf"
    key = (path, size)
    if key not in font_cache:
        font_cache[key] = ImageFont.truetype(path, size)
    return font_cache[key]
print("✅ Fuentes cargadas")


In [ ]:
# ============================================================
# 6. GENERAR LOCUCIÓN (TTS)
# ============================================================
print("Generando audios de locución...")
tts_data = []
timing_starts = [0.0, 2.5, 5.5, 8.5]

for i, te in enumerate(text_elements):
    text = te["content"]
    print(f"  Generando TTS para: '{text}'...")
    tts = gTTS(text, lang="es", tld="com.mx", slow=False)
    path = f"tts_{i}.mp3"
    tts.save(path)
    clip = AudioFileClip(path)
    dur = clip.duration
    start = timing_starts[i] if i < len(timing_starts) else (tts_data[-1]["end"] + 0.5)
    tts_data.append({
        "text": text,
        "start": start,
        "dur": dur,
        "end": start + dur,
        "clip": clip,
    })
    print(f"    {dur:.2f}s (empieza en {start:.1f}s)")

total_dur = max(14.0, max(t["end"] for t in tts_data) + 0.5)
total_dur = total_dur if total_dur <= 14.0 else 14.0
print(f"✅ Locución lista. Duración total: {total_dur:.1f}s")


In [ ]:
# ============================================================
# 7. GENERAR MÚSICA DE FONDO
# ============================================================
print("Generando música de fondo...")
SR = 44100
n = int(total_dur * SR)
t = np.linspace(0, total_dur, n, endpoint=False)

# Acorde C mayor (C4-E4-G4) como pad suave
pad = (np.sin(2*np.pi*261.63*t) + np.sin(2*np.pi*329.63*t) + np.sin(2*np.pi*392.00*t)) / 3 * 0.12
# Línea de bajo simple
bass = np.sin(2*np.pi*130.81*t) * 0.06
# Ritmo: beat cada 0.25s
beat = np.zeros(n)
for b in range(0, n, int(SR * 0.25)):
    beat[b:min(b+800, n)] = 0.25 * np.exp(-np.linspace(0, 5, min(800, n-b)))

music_arr = np.clip(pad + bass + beat, -1, 1)
try:
    music_audio = AudioArrayClip(music_arr, fps=SR).set_duration(total_dur).multiply_volume(0.12)
except Exception:
    # Fallback: write WAV and load with AudioFileClip
    print("  Usando fallback WAV para audio...")
    import struct, wave
    with wave.open("music_bg.wav", "w") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(SR)
        wf.writeframes((music_arr * 32767).astype(np.int16).tobytes())
    music_audio = AudioFileClip("music_bg.wav").set_duration(total_dur).multiply_volume(0.12)
print("✅ Música generada")


In [ ]:
# ============================================================
# 8. RENDERIZAR FRAMES
# ============================================================
print("Renderizando video cuadro por cuadro...")
FPS = 24
n_frames = int(total_dur * FPS)
frames = []
FADE_DUR = 0.35
SLIDE_PX = 40

def render_text_on_frame(draw, te, opacity, slide_off):
    font = get_font(te["weight"], te["font_size"])
    content = te["content"]
    bbox = font.getbbox(content)
    tw = bbox[2] - bbox[0]
    x = te["x"]
    y = te["y"] + slide_off
    if te["align"] == "center":
        x += (te["width"] - tw) / 2
    color = parse_color(te["color"])
    color = tuple(int(c * opacity) for c in color[:3]) + (255,)
    if te["has_shadow"]:
        sc = parse_color(te["shadow_color"])
        sc = tuple(int(c * opacity) for c in sc[:3]) + (int(sc[3] * opacity),)
        draw.text((x + te["shadow_ox"], y + te["shadow_oy"]), content, fill=sc, font=font)
    draw.text((x, y), content, fill=color, font=font)

for fi in range(n_frames):
    t = fi / FPS
    frame = base_frame.copy()
    draw = ImageDraw.Draw(frame)
    for te, td in zip(text_elements, tts_data):
        if td["start"] <= t <= td["end"]:
            if t < td["start"] + FADE_DUR:
                prog = (t - td["start"]) / FADE_DUR
                opacity = prog
                slide_off = SLIDE_PX * (1 - prog)
            else:
                opacity = 1.0
                slide_off = 0
            render_text_on_frame(draw, te, opacity, slide_off)
    frames.append(np.array(frame.convert("RGB")))
    if fi % 48 == 0 or fi == n_frames - 1:
        print(f"  Frame {fi+1}/{n_frames} ({100*(fi+1)//n_frames}%)")
print(f"✅ {len(frames)} frames renderizados")


In [ ]:
# ============================================================
# 9. ARMAR VIDEO + AUDIO
# ============================================================
print("Armando clip de video...")
video_clip = ImageSequenceClip(frames, fps=FPS)

print("Mezclando audio...")
audio_tracks = [music_audio]
for td in tts_data:
    audio_tracks.append(td["clip"].set_start(td["start"]))
final_audio = CompositeAudioClip(audio_tracks)
video_clip = video_clip.set_audio(final_audio)

print("Escribiendo MP4 (esto toma un minuto)...")
OUTPUT = "comerza_animado.mp4"
video_clip.write_videofile(
    OUTPUT,
    codec="libx264",
    audio_codec="aac",
    fps=FPS,
    bitrate="5000k",
    preset="medium",
    threads=2,
    logger=None,
)
print(f"\n✅✅✅ Video generado: {OUTPUT}")
print(f"   Tamaño: {os.path.getsize(OUTPUT) / 1024 / 1024:.1f} MB")


In [ ]:
# ============================================================
# 10. DESCARGAR VIDEO
# ============================================================
from google.colab import files
files.download(OUTPUT)
print("✅ Descarga iniciada — Revisa tu carpeta de Descargas")
